# Section 5 — Outlier Treatment and Final Clean Dataset (15 pts)

## Learning Objectives
- Apply one justified outlier treatment strategy
- Compare dataset before and after full cleaning
- Export `air_quality_clean.csv`

## Your Decisions Log

| Step | Decision | Evidence (plot / table / stat) |
|------|----------|--------------------------------|
|      |          |                                |

## Comparison Table

| Element | Before Cleaning | After Cleaning |
|---------|-----------------|----------------|
| Number of rows | | |
| Number of columns | | |
| Total missing values | | |
| Columns removed | | |
| Variables imputed with mean or median | | |
| Variables imputed with KNNImputer | | |
| Variables treated for outliers | | |

## Tasks
- a) Apply outlier treatment (winsorization, clipping, or justified conservation)
- b) Generate boxplots after treatment
- c) Complete the comparison table above
- d) Save cleaned dataset to `data/processed/air_quality_clean.csv`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.config import RAW_DATA_PATH, MISSING_SENTINEL, KNN_K_VALUES, PROCESSED_DATA_PATH, FIGURES_DIR
from src.load_data import load_raw_air_quality, build_datetime
from src.missing_values import replace_sentinel_with_nan
from src.knn_imputation import select_numerical_columns, knn_impute
from src.clean_pipeline import (
    treat_outliers,
    build_comparison_table,
    export_clean_dataset,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

# Rebuild the earlier stages so the final notebook can run from scratch.
df_before = load_raw_air_quality(RAW_DATA_PATH)
df_before = build_datetime(df_before)
df_before = replace_sentinel_with_nan(df_before, sentinel=MISSING_SENTINEL)
df_before = df_before.dropna(subset=['Datetime']).reset_index(drop=True)
num_cols = select_numerical_columns(df_before)
df_after_imputation = knn_impute(df_before, num_cols, n_neighbors=KNN_K_VALUES[-1])

# Keep the treatment focused on the main pollution variables used in the report.
OUTLIER_COLUMNS = ['CO(GT)', 'C6H6(GT)', 'T', 'RH', 'AH']
TREATMENT_METHOD = 'clip_tukey'

In [ ]:
# Apply the chosen outlier strategy to the selected variables.
df_final = treat_outliers(df_after_imputation, OUTLIER_COLUMNS, method=TREATMENT_METHOD)
display(df_final[OUTLIER_COLUMNS].head())

In [ ]:
# Boxplots after treatment make the effect of clipping easy to see.
fig, axes = plt.subplots(1, len(OUTLIER_COLUMNS), figsize=(4 * len(OUTLIER_COLUMNS), 5))
if len(OUTLIER_COLUMNS) == 1:
    axes = [axes]

for axis, column in zip(axes, OUTLIER_COLUMNS):
    sns.boxplot(y=df_final[column], ax=axis, color='#55a868')
    axis.set_title(column)
    axis.set_xlabel('')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'final_cleaning_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Build the comparison table with the decisions made in the previous steps.
comparison = build_comparison_table(
    df_before,
    df_final,
    metadata={
        'columns_removed': ['NMHC(GT)'],
        'mean_median_imputed': ['CO(GT)', 'C6H6(GT)', 'NOx(GT)', 'NO2(GT)', 'PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH'],
        'knn_imputed': num_cols,
        'outlier_treated': OUTLIER_COLUMNS,
    },
)
display(comparison)

In [ ]:
# Export the final cleaned dataset to the processed folder.
export_clean_dataset(df_final, PROCESSED_DATA_PATH)
print(f'Saved to {PROCESSED_DATA_PATH}')

## Guiding Questions

1. **What changed the most after the complete cleaning process?**

   The biggest changes were the removal of invalid missing-value sentinels, the imputation of incomplete records, and the treatment of outliers. As a result, the distributions became cleaner and less extreme.

2. **Did the final dataset preserve the original information reasonably well?**

   Yes, reasonably well. The core structure and relationships are still there, but some rare extremes and variability were smoothed out during cleaning.

3. **What risks could appear if these cleaning steps were applied automatically without analysis?**

   Automatic cleaning can remove genuine pollution peaks, distort distributions, and introduce bias if the thresholds or imputation rules do not match the behavior of the sensors or the data.